# 📚 28.3 — FREEZING / UNFREEZING

## 🔥 MASTER REVISION NOTES

The core question is:

> **Which pretrained parameters should be allowed to learn on the target task?**

```text
                         PRETRAINED MODEL
                                │
                 ┌──────────────┴──────────────┐
                 ▼                             ▼
          FREEZE PARAMETERS             UNFREEZE PARAMETERS
                 │                             │
                 ▼                             ▼
          Keep weights fixed             Allow weights to update
                 │                             │
                 └──────────────┬──────────────┘
                                ▼
                       PARTIAL FINE-TUNING
```

![Image](https://images.openai.com/static-rsc-4/GWThDTzQcCcSFaXtXPl9LtSdk2-y9T0O1VhK0sVIc-2RnBeJDbpNkUMWcccaIjZkS_gUKH1wmnWrZ1WCx-F_QUZOURr1kFAEUeB7gMX6iVN51WK7MxqsMuX_gIcdUp23G-_pNhNSSNbZoalNxWhZwbD4XCNUnt7nJ1D7j84bGmi2Q31TN7JJRb4JX-HtqEDf?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/gpb83yHhgIZBiNKhHI6ggv0N2UfWFHFiQFLXiShu01drr-gn8uVbTvRtaR1PIVu0XzZgsShMaLOu8wlGnWL_FCZeIkC9qZ5uVQTFcBPMUv9YyHp8cz3mHLMh2S2IuHi7IoLwdy0L30lTSlFM2FFNkDOZ9U3DRznbSeowTBIyfqLec_PXW56G6fsI5jA9_1BZ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/Gc9MmWILVOGKdNxD-ksXukWflpvSPWmeMKp95DOMztQOK16OJxr2hcDNZhwcAFF84hDJiR9n5RQ52VOdsPtlulN0sfEz6kVADzzrJJWTzW--5AmXLLNOzE9LOQd7rWIeemzckkYi0giirI-QZ_WTo9Z8Q_bIHGQp4s1khWsQMS2MBQ3PJhFBJLPIKm2RCWzV?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/uBHgs2ybtzw4dnU_hL2gcUKsI72TMZuMG__td4OVxQULQNRLiVaysQIXl3IwdKGCKdnTv6HmR0dv_YqVhVHnbEKndZM2l8PMZvIL1_fnYMqYmwVCRzteaXm5OQQ0YfXg6vTB_T9LBBsVxRvRKZJZyopOF-HAuam7GR3uQQr_q7yEi1J6AGxu-oA9uLUg_0Sn?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/wk403sn3c50U94e6Rnpz5elk1tiaYxg_jLUpffks7fxjNic9OOVCRUCWffSRTMdaRmy9WNEBFaYDGhEClxIXgrlAfLQSe4ASkntKyAMbKPuGAPewC-aGDyckxkShJISIxJihRSf0vtkMPOndiREjNj767sisHN-eQQpcQ2LKV2Pzk4uSJPNq0Yhdj1xDPLgq?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/tcK-zK4d9uwI6jvQEH_KlQMX5e0kRM9n_MMf_uxA9FuWb-vWN1eBiS803bpnGCn0HA2vGs9l8a77Ltn-zuJD3FuBF5qLjZrhtaOnuDBJuTYoIW0EaNaaOJndf31QEHBpHMRZYQdCC7_ly7TxVQ-xepR-6UkXZ07NY3hmyLl1GC_uE3iusu0m6mfVvXuzwWPH?purpose=fullsize)

---

# 1. 🔒 Frozen Parameters

A **frozen parameter** is a model parameter that is kept unchanged during target-task training.

Normally, gradient descent updates a parameter according to:

$$
W_{\text{new}}
=
W_{\text{old}}
-
\eta
\frac{\partial L}{\partial W}
$$

For a frozen parameter:

$$
\boxed{W_{\text{new}}=W_{\text{old}}}
$$

So the pretrained information stored in that parameter remains unchanged.

### PyTorch

```python
for param in model.parameters():
    param.requires_grad = False
```

---

# 2. 🔓 Trainable Parameters

A **trainable parameter** is allowed to be updated during optimization.

```text
Parameter
   ↓
requires_grad = True
   ↓
Gradient
   ↓
Optimizer
   ↓
Updated parameter
```

For example:

```python
for param in model.layer4.parameters():
    param.requires_grad = True
```

Now `layer4` is trainable.

---

# 3. 🧠 Frozen ≠ Unused

This is extremely important.

Suppose:

```text
Input
  ↓
Frozen layer
  ↓
Trainable layer
  ↓
Loss
```

The frozen layer **still performs its forward computation**.

It produces features that are passed to subsequent layers.

Therefore:

> **Frozen means the parameters aren't updated. It does NOT mean the layer is removed or unused.**

```text
Frozen
  ↓
Still used in forward pass
  ↓
Parameters remain unchanged
```

---

# 4. 🔥 Total Parameters vs Trainable Parameters

Suppose:

```text
Backbone     = 20,000,000 parameters
Classifier   =     10,000 parameters
```

Total:

$$
20,000,000+10,000
=
20,010,000
$$

If the backbone is frozen:

```text
Backbone   → 🔒 20M
Classifier → 🔓 10K
```

Then:

$$
\boxed{\text{Trainable parameters}=10,000}
$$

while:

$$
\boxed{\text{Total parameters}=20,010,000}
$$

### Key distinction

$$
\boxed{\text{Total parameters} \neq \text{Trainable parameters}}
$$

---

# 5. 🧩 Partial Fine-Tuning

### Definition

> **Partial fine-tuning means freezing some pretrained layers while allowing selected pretrained layers to continue learning on the target task.**

Example:

```text
Input
 ↓
Layer 1       🔒
 ↓
Layer 2       🔒
 ↓
Layer 3       🔒
 ↓
Layer 4       🔓
 ↓
Classifier    🔓
 ↓
Output
```

![Image](https://images.openai.com/static-rsc-4/gpb83yHhgIZBiNKhHI6ggv0N2UfWFHFiQFLXiShu01drr-gn8uVbTvRtaR1PIVu0XzZgsShMaLOu8wlGnWL_FCZeIkC9qZ5uVQTFcBPMUv9YyHp8cz3mHLMh2S2IuHi7IoLwdy0L30lTSlFM2FFNkDOZ9U3DRznbSeowTBIyfqLec_PXW56G6fsI5jA9_1BZ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/KyfFX5Cbfuf02WpSLAuA0J_Zg6darwsax9oWQurXuQsHtPl2CM24OVbScthRTP42uzDj4tqMthqA6myqHLY55Dmti9xzlOtbakYru6RA40TtZu4OKMTYxUCjD_tnzNQLhmkVub0JnAzRDjq1vUdYIl_RJ6r6UIx69QSuflQGT5gL85ji4kQGZvHlZfVHxUu_?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/Gc9MmWILVOGKdNxD-ksXukWflpvSPWmeMKp95DOMztQOK16OJxr2hcDNZhwcAFF84hDJiR9n5RQ52VOdsPtlulN0sfEz6kVADzzrJJWTzW--5AmXLLNOzE9LOQd7rWIeemzckkYi0giirI-QZ_WTo9Z8Q_bIHGQp4s1khWsQMS2MBQ3PJhFBJLPIKm2RCWzV?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/FCJOm4VXygGZOLxvuY8f4fSrFpHlARdYkPiCSQueIt1JkvGUp3kBKynWF0NzQJnKHvp_vLHlMsAvV6q3iB5XmjRMbvm8fDaX8h6YSp_etpkiwTD5rd0-MsBervIZ6UTKBi0RaCjH1Sa2YOEr2p6SPoKDRtuiUWEl708l7c06cvHvCFyZw8dvUv80PnxhEsk0?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/wk403sn3c50U94e6Rnpz5elk1tiaYxg_jLUpffks7fxjNic9OOVCRUCWffSRTMdaRmy9WNEBFaYDGhEClxIXgrlAfLQSe4ASkntKyAMbKPuGAPewC-aGDyckxkShJISIxJihRSf0vtkMPOndiREjNj767sisHN-eQQpcQ2LKV2Pzk4uSJPNq0Yhdj1xDPLgq?purpose=fullsize)

---

# 6. 🧠 Why Freeze Early Layers?

CNNs learn hierarchical representations.

A simplified hierarchy:

```text
Early layers
    ↓
Edges
Corners
Simple textures
       ↓
Middle layers
    ↓
Patterns
Shapes
Complex textures
       ↓
Later layers
    ↓
Object parts
Higher-level representations
```

Early layers can contain relatively general visual features, while later layers can be more specialized to the original task.

Therefore, a common strategy is:

```text
Early layers → 🔒
Later layers → 🔓
Classifier   → 🔓
```

⚠️ **This is a strategy, not a universal rule.**

The appropriate amount of unfreezing depends on the source task, target task, dataset size, and experimental results.

---

# 7. 🧮 Numerical Example

Suppose a CNN contains:

| Layer      | Parameters | Status |
| ---------- | ---------: | ------ |
| Layer 1    |         1M | 🔒     |
| Layer 2    |         2M | 🔒     |
| Layer 3    |         4M | 🔒     |
| Layer 4    |         8M | 🔓     |
| Classifier |       0.1M | 🔓     |

Total:

$$
1+2+4+8+0.1=15.1M
$$

Trainable:

$$
8+0.1=\boxed{8.1M}
$$

Therefore:

```text
Total model parameters     = 15.1M
Trainable parameters       = 8.1M
Frozen parameters          = 7.0M
```

---

# 8. 💻 PyTorch — Freeze Everything

```python
from torchvision.models import resnet50, ResNet50_Weights

model = resnet50(
    weights=ResNet50_Weights.DEFAULT
)

for param in model.parameters():
    param.requires_grad = False
```

Now:

```text
layer1 → 🔒
layer2 → 🔒
layer3 → 🔒
layer4 → 🔒
fc     → 🔒
```

This is the starting point for feature extraction.

---

# 9. 💻 PyTorch — Unfreeze Selected Layers

Suppose we want to fine-tune `layer4`:

```python
for param in model.layer4.parameters():
    param.requires_grad = True
```

Then replace the classifier:

```python
import torch.nn as nn

model.fc = nn.Linear(
    model.fc.in_features,
    num_classes
)
```

Now:

```text
layer1 → 🔒
layer2 → 🔒
layer3 → 🔒
layer4 → 🔓
fc     → 🔓
```

This is **partial fine-tuning**.

---

# 10. 🔥 Learning Rate and Unfreezing

The newly initialized classifier and pretrained layers can have different starting points.

```text
New classifier
      ↓
Randomly initialized
      ↓
Needs substantial learning

Pretrained layer
      ↓
Already contains useful representation
      ↓
Needs careful adaptation
```

Therefore, it can be useful to use **different learning rates**.

Example:

```python
optimizer = torch.optim.AdamW([
    {
        "params": model.layer4.parameters(),
        "lr": 1e-5
    },
    {
        "params": model.fc.parameters(),
        "lr": 1e-3
    }
])
```

Conceptually:

```text
layer4 → 🔓 smaller LR
fc     → 🔓 larger LR
```

The exact values are **not universal**.

---

# 11. 🧠 Why Different Learning Rates?

For a newly initialized classifier:

$$
W_{\text{new}}
$$

doesn't contain useful target-task knowledge yet.

For pretrained layers:

$$
W_{\text{pretrained}}
$$

already contains useful representations.

So we often want:

$$
\boxed{\text{larger updates for the new head}}
$$

and:

$$
\boxed{\text{smaller updates for pretrained layers}}
$$

when using differential learning rates.

---

# 12. 🧩 Three Levels of Adaptation

```text
                 PRETRAINED MODEL
                        │
        ┌───────────────┼───────────────┐
        ▼               ▼               ▼
 Feature Extraction  Partial FT      Full FT
        │               │               │
        ▼               ▼               ▼
   🔒🔒🔒🔒           🔒🔒🔓🔓         🔓🔓🔓🔓
        │               │               │
       Head            Head            Head
        🔓               🔓               🔓
```

### Feature extraction

Only the new classifier learns.

### Partial fine-tuning

Some pretrained layers + classifier learn.

### Full fine-tuning

The entire model can adapt.

---

# 13. 🎯 Partial Fine-Tuning as a Control Knob

Think of unfreezing as controlling the amount of model adaptation:

```text
Less adaptation
      ↑
      │
Feature extraction
      │
Partial fine-tuning
      │
Full fine-tuning
      │
      ↓
More adaptation
```

More trainable parameters means the model has more freedom to adapt.

But it also means:

* more computation
* more optimization
* more memory requirements during training
* potentially greater overfitting risk

So:

> **More trainable parameters ≠ automatically better performance.**

---

# 14. 🔬 Practical Workflow

A useful workflow is:

```text
Pretrained model
       ↓
Replace classifier
       ↓
Freeze backbone
       ↓
Train classifier
       ↓
Evaluate
       ↓
Need more adaptation?
       ↓
Unfreeze selected layers
       ↓
Fine-tune with suitable LR
       ↓
Evaluate again
```

This gives you a controlled experimental progression.

---

# 15. 🧪 What to Measure?

When experimenting with different freezing strategies, monitor:

### Model performance

* Training loss
* Validation loss
* Training accuracy
* Validation accuracy
* Per-class metrics where appropriate

### Training behavior

* Convergence
* Overfitting
* Number of epochs required

### Resources

* Trainable parameter count
* GPU memory
* Training time
* Inference characteristics

The goal is to understand:

> **How much additional adaptation do I get by making more parameters trainable, and what does it cost?**

---

# 16. 🚨 Common Mistakes

### ❌ Mistake 1 — "Frozen layers are removed."

No.

```text
Frozen ≠ removed
```

They still participate in the forward pass.

---

### ❌ Mistake 2 — Confusing total and trainable parameters

Remember:

$$
\boxed{
\text{Total}
=
\text{Frozen}
+
\text{Trainable}
}
$$

---

### ❌ Mistake 3 — Unfreezing but forgetting the optimizer

The optimizer must be configured so that the parameters you intend to train are actually optimized.

A useful pattern:

```python
optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-4
)
```

---

### ❌ Mistake 4 — Assuming later layers must always be trained

Not necessarily.

The appropriate choice depends on:

* source-target similarity
* target dataset size
* computational budget
* validation behavior

---

### ❌ Mistake 5 — Assuming more trainable layers means better results

More trainable layers provide more adaptation capacity, but they also change the optimization problem and resource requirements.

**Experiment and measure.**

---

# 17. 🎯 Interview Questions

### Q1. What is a frozen parameter?

A parameter that is kept unchanged during target-task training.

### Q2. What is a trainable parameter?

A parameter that can receive gradient-based updates during optimization.

### Q3. What is partial fine-tuning?

Freezing some pretrained layers while allowing selected pretrained layers and the task-specific head to learn.

### Q4. Why are early layers often frozen?

They often contain relatively general visual representations such as edges, corners, and textures.

### Q5. Can a frozen layer participate in the forward pass?

**Yes.**

### Q6. Can different layers have different learning rates?

**Yes.** Optimizer parameter groups allow different learning rates.

---

# 🧠 18. MASTER MEMORY DIAGRAM

```text
                    PRETRAINED MODEL
                           │
                           ▼
                "Which parameters learn?"
                           │
          ┌────────────────┼────────────────┐
          ▼                ▼                ▼
     Freeze all       Freeze some      Unfreeze all
          │                │                │
          ▼                ▼                ▼
 Feature Extraction   Partial FT        Full FT
          │                │                │
          ▼                ▼                ▼
       🔒🔒🔒           🔒🔒🔓           🔓🔓🔓
          │                │                │
          └────────────────┼────────────────┘
                           ▼
                     Target classifier
                           🔓
```

### 🔑 Three things to remember

> **Frozen parameter → doesn't update.**

> **Trainable parameter → can update through optimization.**

> **Partial fine-tuning → freeze some layers, unfreeze selected layers.**

---

# ⚡ 30-SECOND REVISION

```text
Frozen
→ requires_grad=False
→ parameter doesn't update

Trainable
→ requires_grad=True
→ optimizer can update it

Partial fine-tuning
→ some layers 🔒
→ selected layers 🔓
→ classifier 🔓

Common pattern
→ early layers 🔒
→ later layers 🔓
→ classifier 🔓

Different LR?
→ Yes, using optimizer parameter groups
```

---

## 🔗 Official PyTorch Reference

PyTorch's official transfer-learning tutorial demonstrates both **fine-tuning a pretrained ConvNet** and using a pretrained ConvNet as a **fixed feature extractor**.

[PyTorch — Transfer Learning for Computer Vision](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html?utm_source=chatgpt.com)

---

